# EDA (I)
## Context
- Two datasets need to be explored: bus routes (csv), bus event time (parquet)
- From now on bus event time dataset will be referred to as Main Dataset as it'd the primary source of data for the ML modeling. 
## Goal
- Fully explored the two datasets, check for data integrity, missing values, any discrepancies
- Determine the features that would go into the ML modeling

# Import

In [1]:
# to import from local modules
from pathlib import Path
import sys

parent_dir = str(Path().resolve().parents[0])
sys.path.insert(0, parent_dir)

In [2]:
import polars as pl
import plotly.express as px
from datetime import datetime, date, timedelta, time
from src.constants import DATA_FILE, ROUTES
from src.helpers import clean_df

pl.Config.set_tbl_rows(
    30
)  # to render a full trip (eg. from stop 1 to stop 30 for instance)

polars.config.Config

In [3]:
df_full = pl.scan_parquet(DATA_FILE)
df_routes = pl.read_csv(ROUTES)
df_full.head(5).collect().glimpse()

Rows: 5
Columns: 29
$ PlateNumb         <str> 'KKA-3975', '396-FQ', '396-FQ', '396-FQ', '396-FQ'
$ OperatorID        <i32> 16, 35, 35, 35, 35
$ OperatorNo        <i32> 702, 1308, 1308, 1308, 1308
$ RouteUID          <str> 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968'
$ RouteID           <i32> 968, 968, 968, 968, 968
$ RouteNameZh_tw    <str> '0968', '0968', '0968', '0968', '0968'
$ RouteNameEn       <str> '0968', '0968', '0968', '0968', '0968'
$ SubRouteUID       <str> 'THB096802', 'THB096802', 'THB096802', 'THB096802', 'THB096802'
$ SubRouteID        <str> '096802', '096802', '096802', '096802', '096802'
$ SubRouteNameZh_tw <str> '0968', '0968', '0968', '0968', '0968'
$ SubRouteNameEn    <str> '0968', '0968', '0968', '0968', '0968'
$ Direction         <cat> 返程, 返程, 返程, 返程, 返程
$ StopUID           <str> 'THB300178', 'THB300198', 'THB300198', 'THB300197', 'THB300197'
$ StopID            <i32> 300178, 300198, 300198, 300197, 300197
$ StopNameZh_tw     <str> '大竹消防隊', '庫倫街口', '庫倫街口'

# Routes discrepancy

In [4]:
print(
    f"Routes csv has {df_routes.select(pl.col('SubRouteID').n_unique()).item()} unique bus routes"
)
print(
    f"Main dataset has {df_full.select(pl.col('SubRouteID').n_unique()).collect().item()} unique bus routes"
)

Routes csv has 1781 unique bus routes
Main dataset has 1801 unique bus routes


- There is a mismatch between the number of bus routes. 
- What are the mismatched stops (stops that only appear in one of the datasets)?

In [5]:
# Instead of list comprehension, use Polars native set logic:
routes_a = df_routes.select("SubRouteID").unique().to_series().to_list()
routes_b = (
    pl.scan_parquet(DATA_FILE)
    .select("SubRouteID")
    .unique()
    .collect()
    .to_series()
    .to_list()
)

# Identify routes that are ONLY in one of the datasets
discrepancy = set(routes_a).symmetric_difference(set(routes_b))
print(sorted(discrepancy))


['114102', '121201', '121202', '1612A1', '1612B1', '1613B1', '1613B2', '161501', '161502', '1615A2', '161601', '161602', '1618C2', '1618D1', '1618E2', '1621B2', '1621C2', '162802', '1628A1', '1628A2', '1633A2', '1633D1', '1633D2', '1633E1', '1633E2', '1635A1', '1635A2', '1635C1', '1635C2', '1636A1', '1636A2', '1637A1', '1637A2', '1650A1', '1650A2', '1650B1', '1650B2', '1650C1', '1650C2', '1651B1', '1651B2', '1651C1', '1651C2', '1800A1', '1800A2', '180101', '1813C2', '1822A2', '183002', '1832B1', '1832B2', '1832C2', '1832D2', '1838B1', '1838B2', '1842A1', '1842A2', '1862B1', '1866A1', '200101', '200102', '560501', '560502', '561301', '561302', '562201', '562202', '563501', '563502', '666201', '666202', '672301', '672302', '672401', '672402', '691901', '7000G1', '7000G2', '7000H1', '7000H2', '7001B1', '7001B2', '7002A1', '7002A2', '700502', '7127A1', '7127A2', '7204A2', '7209I2', '7217B2', '7319G2', '7500N2', '7500U2', '750201', '750202', '8101D1', '813201', '813202', '8168A2', '8168B1',

## Finding
- Based on external sources, these routes are either
    - newly added
    - recently stopped
    - renamed (re-indexed)
- For final modeling, we should filter out any stopped bus routes.
- If existing data of the newly added routes is not sufficient for training, they should also be filtered out.
    - So I should implement some logic/threshold to filter out routes that have too few trips

# Main Dataset EDA
## Initial exploration and cleaning

In [6]:
df_full.head(20).collect().glimpse()

Rows: 20
Columns: 29
$ PlateNumb         <str> 'KKA-3975', '396-FQ', '396-FQ', '396-FQ', '396-FQ', '396-FQ', '396-FQ', 'KKA-3975', 'KKA-3975', 'KKA-3975'
$ OperatorID        <i32> 16, 35, 35, 35, 35, 35, 35, 16, 16, 16
$ OperatorNo        <i32> 702, 1308, 1308, 1308, 1308, 1308, 1308, 702, 702, 702
$ RouteUID          <str> 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968'
$ RouteID           <i32> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameZh_tw    <str> '0968', '0968', '0968', '0968', '0968', '0968', '0968', '0968', '0968', '0968'
$ RouteNameEn       <str> '0968', '0968', '0968', '0968', '0968', '0968', '0968', '0968', '0968', '0968'
$ SubRouteUID       <str> 'THB096802', 'THB096802', 'THB096802', 'THB096802', 'THB096802', 'THB096801', 'THB096802', 'THB096801', 'THB096801', 'THB096802'
$ SubRouteID        <str> '096802', '096802', '096802', '096802', '096802', '096801', '096802', '096801', '096801', '096802'


- There are multiple columns for similar information (eg. RouteUID vs RouteID vs RouteNameZh_tw vs RouteNameEn and more)
- Mutiple time columns following ISO 8601 format => needs parsed into python datetime

## Time span

In [7]:
df_full.filter(pl.col("GPSTime") != "2").select(
    pl.min("GPSTime").alias("min"), pl.max("GPSTime").alias("max")
).collect()

min,max
str,str
"""2025-02-28T00:00:00+08:00""","""2026-02-27T23:59:58+08:00"""


- Data spans from 2025-2-28 to 2026-2-27 (a full year)

## All columns inspection

In [8]:
# in ETL, every day has the exact same schema and data patterns
# here for efficiency, use one day of data only
df_full.filter(pl.col("GPSTime").str.starts_with("2025-02-28")).drop(
    [
        "GPSTime",
        "TripStartTime",
        "TransTime",
        "SrcRecTime",
        "SrcTransTime",
        "SrcUpdateTime",
        "UpdateTime",
    ]
).select(pl.all().n_unique()).collect()

PlateNumb,OperatorID,OperatorNo,RouteUID,RouteID,RouteNameZh_tw,RouteNameEn,SubRouteUID,SubRouteID,SubRouteNameZh_tw,SubRouteNameEn,Direction,StopUID,StopID,StopNameZh_tw,StopNameEn,StopSequence,MessageType,DutyStatus,BusStatus,A2EventType,TripStartTimeType
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2331,47,47,467,467,467,467,1326,1326,701,701,2,18327,18327,6573,6955,212,1,3,5,2,3


- Finding
    - 1326 SubRouteID vs 467 RouteID: SubRouteID contains extra information like route, direction, sub-route etc. 
    - 6573 StopNameZh_tw vs 6955 StopNameEn: There exist typos in StopNameEn

# Data cleaning
- Write a reusable function that outputs a clean dataset
- Strip away unnecessary features
- Features kept: GPSTime, SubRouteID, StopNameZh_tw, Direction, A2EventType (arriving vs departing)
- Parse GPSTime into python datetime objects

In [10]:
# Try with a specific route first before scaling to entire dataset
TARGET_ROUTE = "17280"

In [11]:
df = (
    df_full.filter(
        pl.col("SubRouteID").str.starts_with(TARGET_ROUTE),
        pl.col("TripStartTimeType") == "實際發車時間",
    )
    .with_columns(
        pl.col("GPSTime")
        .str.replace(r"\+08:00", "")
        .str.to_datetime(format="%Y-%m-%dT%H:%M:%S")
        .alias("Time"),
        # The last digit of SubRouteID represents same info from Direction
        # Separating them would be clearer
        pl.col("SubRouteID").str.replace(r".$", "").alias("Route"),
    )
    .select(
        pl.col("Route"),
        pl.col("PlateNumb").alias("Plate"),
        pl.col("Direction"),
        pl.col("StopNameZh_tw").alias("Stop"),
        pl.col("A2EventType").alias("Event"),
        pl.col("Time"),
    )
    .filter(
        pl.col("Time").dt.date() == date(2026, 2, 25),
    )
    .collect()
)
df.glimpse()

Rows: 1795
Columns: 6
$ Route              <str> '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280'
$ Plate              <str> '057-FS', '057-FS', '057-FS', 'KKB-2059', '057-FS', 'KKB-2057', '057-FS', 'KKB-1713', 'KKB-2670', '057-FS'
$ Direction          <cat> 返程, 返程, 返程, 返程, 返程, 去程, 返程, 去程, 返程, 去程
$ Stop               <str> '幸安國小', '幸安國小', '仁愛建國路口一', '清華大學', '仁愛建國路口一', '捷運公館站', '捷運景美站', '龍潭運動公園', '科技生活館', '師大分部'
$ Event              <cat> 離站, 進站, 進站, 進站, 離站, 進站, 離站, 離站, 離站, 離站
$ Time      <datetime[μs]> 2026-02-25 16:15:27, 2026-02-25 16:15:14, 2026-02-25 16:15:32, 2026-02-25 16:15:38, 2026-02-25 16:15:50, 2026-02-25 16:15:59, 2026-02-25 21:21:14, 2026-02-25 21:21:34, 2026-02-25 21:21:02, 2026-02-25 11:22:13



- This function is put in src.helpers for later reuse.

# EDA of Single Route (1728)

## First stop

In [13]:
df.filter(pl.col("Stop") == "新竹轉運站", pl.col("Direction") == "返程").sort("Time")

Route,Plate,Direction,Stop,Event,Time
str,str,cat,str,cat,datetime[μs]
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:15:46
"""17280""","""KKB-2669""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:40:54
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:02:19
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:20:04
"""17280""","""057-FS""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:42:05
"""17280""","""KKB-2035""","""返程""","""新竹轉運站""","""離站""",2026-02-25 07:01:33
"""17280""","""KKB-2055""","""返程""","""新竹轉運站""","""離站""",2026-02-25 08:01:17
"""17280""","""KKB-2059""","""返程""","""新竹轉運站""","""離站""",2026-02-25 09:01:20
"""17280""","""KKB-2670""","""返程""","""新竹轉運站""","""離站""",2026-02-25 10:03:37


- Finding
    - For this route, there are 22 trips departing from the first stop. (which matches external sources)
    - All data columns are clean.

##  Explore `Event`

In [14]:
fig = px.bar(
    df.filter(pl.col("Direction") == "返程")
    .group_by(["Stop", "Event"])
    .len()
    .sort(["Stop", "Event"]),
    y="Stop",
    x="len",
    color="Event",
)
fig.show()
# fig.show(renderer = "browser") to see in details

- Uneven distribution of bus records at each stop. Deeper investigation on two specific stops to compare the records.

In [15]:
depart = (
    df.filter(
        pl.col("Event") == "離站",
        pl.col("Stop") == "新竹轉運站",
        pl.col("Direction") == "返程",
    )
    .with_columns(pl.col("Time").alias("DepartureTime"))
    .sort("Time")
)
depart

Route,Plate,Direction,Stop,Event,Time,DepartureTime
str,str,cat,str,cat,datetime[μs],datetime[μs]
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:15:46,2026-02-25 05:15:46
"""17280""","""KKB-2669""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:40:54,2026-02-25 05:40:54
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:02:19,2026-02-25 06:02:19
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:20:04,2026-02-25 06:20:04
"""17280""","""057-FS""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:42:05,2026-02-25 06:42:05
"""17280""","""KKB-2035""","""返程""","""新竹轉運站""","""離站""",2026-02-25 07:01:33,2026-02-25 07:01:33
"""17280""","""KKB-2055""","""返程""","""新竹轉運站""","""離站""",2026-02-25 08:01:17,2026-02-25 08:01:17
"""17280""","""KKB-2059""","""返程""","""新竹轉運站""","""離站""",2026-02-25 09:01:20,2026-02-25 09:01:20
"""17280""","""KKB-2670""","""返程""","""新竹轉運站""","""離站""",2026-02-25 10:03:37,2026-02-25 10:03:37


In [16]:
destination = (
    df.filter(
        pl.col("Event") == "進站",
        pl.col("Stop") == "龍潭運動公園",
        pl.col("Direction") == "返程",
    )
    .with_columns(pl.col("Time").alias("ArrivalTime"))
    .sort("Time")
)
destination

Route,Plate,Direction,Stop,Event,Time,ArrivalTime
str,str,cat,str,cat,datetime[μs],datetime[μs]
"""17280""","""KKB-2058""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:00:09,2026-02-25 06:00:09
"""17280""","""KKB-2669""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:23:58,2026-02-25 06:23:58
"""17280""","""KKB-1715""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:49:51,2026-02-25 06:49:51
"""17280""","""KKB-2057""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:02:17,2026-02-25 07:02:17
"""17280""","""057-FS""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:35:25,2026-02-25 07:35:25
"""17280""","""KKB-2035""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:50:22,2026-02-25 07:50:22
"""17280""","""KKB-2055""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 09:12:24,2026-02-25 09:12:24
"""17280""","""KKB-2059""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 10:08:11,2026-02-25 10:08:11
"""17280""","""KKB-2670""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 10:59:46,2026-02-25 10:59:46


## Finding
- There are 22 trips that depart from "新竹轉運站", among which 21 trips are recorded to pass through "龍潭運動公園" (a middle stop)
- Only '離站' (departure) is recorded for the first stop.
- Only '進站' (arriving) is recorded for the last stop.
- The middle stops have different number of records, which means we **DON'T** have complete data of when each bus reaches each stop.
- Explanation 
    - Only when a passenger gets on/off at a stop would the time be recorded.
    - Therefore, the less popular a stop is, the more missing time data there are.
- **Problem**: We can't reliably calculate the travel time between each stop. Some unpopular stops will have a lot of missing values.
- Solution: For the final model, we have to only pick certain stops (by some threshold) and make predictions of travel time between those stops. If a user wants to predict time between two stops, the final application should first predict the time between all the intermediate stops and add up the predictions before giving the user a clean prediction.

## Check other patterns
- From above, we know that some stops are not recorded in the middle stops.
- What about the very first stop?

In [18]:
# check if the above finding the true for all the dates
min_date = date(2026, 1, 1)
max_date = date(2026, 2, 25)
diff = (max_date - min_date).days
dates = [min_date + timedelta(days=x) for x in range(diff)]

df = (
    df_full.filter(pl.col("SubRouteID").str.starts_with("1728"))
    .pipe(clean_df)
    .filter(pl.col("Time").is_between(min_date, max_date))
    .sort("Time")
    .collect(engine="streaming")
)


df.group_by(pl.col("Time").dt.date().alias("date")).agg(
    [
        pl.col("Event")
        .filter(
            (pl.col("Event") == "離站")
            & (pl.col("Stop") == "新竹轉運站")
            & (pl.col("Direction") == "返程")
        )
        .len()
        .alias("num_departure"),
        pl.col("Event")
        .filter(
            (pl.col("Event") == "進站")
            & (pl.col("Stop") == "龍潭運動公園")
            & (pl.col("Direction") == "返程")
        )
        .len()
        .alias("num_destination"),
    ]
).filter(pl.col("num_departure") < pl.col("num_destination")).sort("date")


date,num_departure,num_destination
date,u32,u32
2026-01-01,11,12
2026-01-07,21,22
2026-02-08,19,20


## Finding
- Turns out 3 out of the testing range has fewer rows for departing trips than arrival trips. 
- This means some trips are not even recorded for the first stop of the trip.

# Calculate travel time between two stops
- Before scaling to all other routes/stops, first try a single route with two specific stops.

In [22]:
temp = df.filter(
    pl.col("Time").is_between(date(2026, 1, 1), datetime(2026, 1, 1, 23, 59, 59))
)

temp_depart = temp.filter(
    pl.col("Event") == "離站",
    pl.col("Stop") == "新竹轉運站",
    pl.col("Direction") == "返程",
)

temp_dest = temp.filter(
    pl.col("Event") == "進站",
    pl.col("Stop") == "龍潭運動公園",
    pl.col("Direction") == "返程",
)

display(temp_depart)
display(temp_dest)

Route,Plate,Direction,Stop,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,cat,datetime[μs],cat
"""17280""","""057-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 06:01:45,"""Thu"""
"""17280""","""056-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 07:00:57,"""Thu"""
"""17280""","""KKA-1832""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 10:04:33,"""Thu"""
"""17280""","""056-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 12:01:19,"""Thu"""
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 13:00:43,"""Thu"""
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 15:00:06,"""Thu"""
"""17280""","""KKB-2056""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 16:03:52,"""Thu"""
"""17280""","""057-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 17:02:48,"""Thu"""
"""17280""","""058-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 18:01:29,"""Thu"""


Route,Plate,Direction,Stop,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,cat,datetime[μs],cat
"""17280""","""057-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 06:54:56,"""Thu"""
"""17280""","""058-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 08:56:05,"""Thu"""
"""17280""","""KKA-1832""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 10:56:17,"""Thu"""
"""17280""","""057-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 12:08:23,"""Thu"""
"""17280""","""056-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 12:53:47,"""Thu"""
"""17280""","""KKB-2057""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 15:56:15,"""Thu"""
"""17280""","""KKB-2056""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 17:02:50,"""Thu"""
"""17280""","""057-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 18:17:37,"""Thu"""
"""17280""","""058-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 19:02:19,"""Thu"""


## Fuzzy join (`join_asof`)
- Use the `Plate` to identify which two records belong to the same trip. 
- Since the times of two records are obviously not the same, we match by tolerating a 2-hour difference, which seems to work in this case.
- If there's no match (due to problems demonstrated above), drop the rows.
- Create a `Duration_min` column (this will be the labels in ML modeling)
- Future Consideration
    - To scale to other routes, there has to be a way to automatically assign `tolerance` value depending on the stops
    - To scale to entire dataset, we should also group by not just `Plate` but also `Route`

In [25]:
result = (
    depart.join_asof(
        destination,
        on="Time",
        by="Plate",
        strategy="forward",
        check_sortedness=False,
        tolerance="2h",
    )
    .drop_nulls()
    .with_columns(
        (
            (pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds() / 60
        ).alias("Duration_min")
    )
)
result

Route,Plate,Direction,Stop,Event,Time,DepartureTime,Route_right,Direction_right,Stop_right,Event_right,ArrivalTime,Duration_min
str,str,cat,str,cat,datetime[μs],datetime[μs],str,cat,str,cat,datetime[μs],f64
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:15:46,2026-02-25 05:15:46,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:00:09,44.383333
"""17280""","""KKB-2669""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:40:54,2026-02-25 05:40:54,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:23:58,43.066667
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:02:19,2026-02-25 06:02:19,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:49:51,47.533333
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:20:04,2026-02-25 06:20:04,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:02:17,42.216667
"""17280""","""057-FS""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:42:05,2026-02-25 06:42:05,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:35:25,53.333333
"""17280""","""KKB-2035""","""返程""","""新竹轉運站""","""離站""",2026-02-25 07:01:33,2026-02-25 07:01:33,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:50:22,48.816667
"""17280""","""KKB-2055""","""返程""","""新竹轉運站""","""離站""",2026-02-25 08:01:17,2026-02-25 08:01:17,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 09:12:24,71.116667
"""17280""","""KKB-2059""","""返程""","""新竹轉運站""","""離站""",2026-02-25 09:01:20,2026-02-25 09:01:20,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 10:08:11,66.85
"""17280""","""KKB-2670""","""返程""","""新竹轉運站""","""離站""",2026-02-25 10:03:37,2026-02-25 10:03:37,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 10:59:46,56.15


In [ ]:
# plot the travel time distribution
px.bar(result, x="Time", y="Duration_min")

## Finding
- Clear morning rush hour and evening rush hour pattern

## More plotting to identify patterns
- The following function automates the above procedures to more conveniently plot travel times of different days

In [26]:
def duration_plot_by_date(
    df: pl.DataFrame, d: date, depart_stop: str, dest_stop: str, direction: str
) -> None:
    try:
        temp = df.filter(
            pl.col("Time").is_between(d, datetime(d.year, d.month, d.day, 23, 59, 59))
        )

        depart_df = (
            temp.filter(
                pl.col("Event") == "離站",
                pl.col("Stop") == depart_stop,
                pl.col("Direction") == direction,
            )
            .with_columns(pl.col("Time").alias("DepartureTime"))
            .sort("Time")
        )
        dest_df = (
            temp.filter(
                pl.col("Event") == "進站",
                pl.col("Stop") == dest_stop,
                pl.col("Direction") == direction,
            )
            .with_columns(pl.col("Time").alias("ArrivalTime"))
            .sort("Time")
        )

        result_df = (
            depart_df.join_asof(
                dest_df,
                on="Time",
                by="Plate",
                strategy="forward",
                tolerance="2h",
                check_sortedness=False,
            )
            .drop_nulls()
            .with_columns(
                (
                    (pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds()
                    / 60
                ).alias("Duration_min")
            )
        )
        count = result_df.height
        px.bar(
            result_df,
            x="Time",
            y="Duration_min",
            title=f"{d.month}-{d.day}({d.strftime('%A')})",
            subtitle=f"{count} trips in total",
        ).show()
    except Exception as e:
        print(f"Error: {e}")

In [27]:
# investigate travel time distribution of every Tuesday
min_date = date(2025, 9, 30)
max_date = date(2025, 12, 31)
diff = (max_date - min_date).days
dates = [min_date + timedelta(days=x) for x in range(0, diff, 7)]

df = (
    df_full.filter(pl.col("SubRouteID").str.starts_with("1728"))
    .pipe(clean_df)
    .filter(pl.col("Time").is_between(min_date, max_date))
    .sort("Time")
    .collect(engine="streaming")
)

for d in dates:
    duration_plot_by_date(df, d, "清華大學", "龍潭運動公園", "返程")

## Finding Summary (after plotting several combinations)
1. Significant interaction effects between **time** and **weekday**
2. For this bus route:
    - Morning rush hours roughly 7:30 - 9:30
    - Evening rush hours roughly 16:30 - 20:00
3. Clearly different patterns between weekdays and weekends
    - Weekdays: morning rush hours and evening rush hours
    - Weekends: Generally longer travel time throughout the whole day. Less rush hour effects
3. Occasionally, huge traffic time (potentially due to weather reasons)

# Calculate travel time of all dates
- Try single route all days to see if the function scales

In [28]:
df = (
    df_full.filter(pl.col("SubRouteID").str.starts_with("1728"))
    .pipe(clean_df)
    .sort("Time")
    .collect(engine="streaming")
)

In [29]:
temp = df.with_columns(pl.col("Time").dt.date().alias("Date"))
depart_df = (
    temp.filter(
        pl.col("Event") == "離站",
        pl.col("Stop") == "新竹轉運站",
        pl.col("Direction") == "返程",
    )
    .with_columns(pl.col("Time").alias("DepartureTime"))
    .sort("Time")
)
dest_df = (
    temp.filter(
        pl.col("Event") == "進站",
        pl.col("Stop") == "龍潭運動公園",
        pl.col("Direction") == "返程",
    )
    .with_columns(pl.col("Time").alias("ArrivalTime"))
    .sort("Time")
)
depart_df.join_asof(
    dest_df,
    on="Time",
    by=["Date", "Plate"],
    strategy="forward",
    tolerance="2h",
    check_sortedness=False,
).drop_nulls().with_columns(
    ((pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds() / 60).alias(
        "Duration_min"
    )
)


Route,Plate,Direction,Stop,StopSeq,Event,Time,Day_of_week,Date,DepartureTime,Route_right,Direction_right,Stop_right,StopSeq_right,Event_right,Day_of_week_right,ArrivalTime,Duration_min
str,str,cat,str,i32,cat,datetime[μs],cat,date,datetime[μs],str,cat,str,i32,cat,cat,datetime[μs],f64
"""17280""","""058-FS""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 06:01:29,"""Fri""",2025-02-28,2025-02-28 06:01:29,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 06:45:15,43.766667
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 06:59:02,"""Fri""",2025-02-28,2025-02-28 06:59:02,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 07:41:21,42.316667
"""17280""","""KKB-2035""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 08:01:19,"""Fri""",2025-02-28,2025-02-28 08:01:19,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 08:52:13,50.9
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 09:00:50,"""Fri""",2025-02-28,2025-02-28 09:00:50,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 10:06:22,65.533333
"""17280""","""KKB-2056""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 10:28:42,"""Fri""",2025-02-28,2025-02-28 10:28:42,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 11:40:29,71.783333
"""17280""","""058-FS""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 11:38:58,"""Fri""",2025-02-28,2025-02-28 11:38:58,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 12:38:16,59.3
"""17280""","""KKB-1713""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 12:01:27,"""Fri""",2025-02-28,2025-02-28 12:01:27,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 13:02:35,61.133333
"""17280""","""KKB-1692""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 13:01:40,"""Fri""",2025-02-28,2025-02-28 13:01:40,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 14:00:14,58.566667
"""17280""","""KKB-2059""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 14:02:01,"""Fri""",2025-02-28,2025-02-28 14:02:01,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 15:16:33,74.533333


## Finding
- Based on some simple calculations/inspections, the vast majority of calculations work.
- There do exist some anomalies, which would have to be separated out eventually.